[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rslab-ntua/MSc_GBDA/blob/master/2025/GBDA_2025_Lab3.ipynb)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.models import resnet18, ResNet18_Weights
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from datasets import load_dataset

# Load EuroSAT Dataset from Huggingface

In [ ]:
# Load EuroSAT RGB dataset from Hugging Face
dataset = load_dataset("blanchon/EuroSAT_RGB")
print("Dataset loaded successfully!")

# Check dataset structure
print(dataset)

# Get class names
class_names = dataset["train"].features["label"].names  # type: ignore
print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")

# Count images per class
class_counts = {}
for example in dataset["train"]:  # type: ignore
    label = class_names[example["label"]]  # type: ignore
    if label in class_counts:
        class_counts[label] += 1
    else:
        class_counts[label] = 1

for cls, count in class_counts.items():
    print(f"{cls}: {count} images")

# Visualize sample images

In [ ]:
# Visualize sample images from each class
plt.figure(figsize=(15, 10))
class_examples = {}

# Find one example of each class
for example in dataset["train"]:  # type: ignore
    label = class_names[example["label"]]  # type: ignore
    if label not in class_examples:
        class_examples[label] = example["image"]  # type: ignore
    if len(class_examples) == len(class_names):
        break

# Plot one example from each class
for i, (cls, img) in enumerate(class_examples.items()):
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")
plt.tight_layout()
plt.show()

# Create PyTorch Dataset and DataLoaders

In [ ]:
# Create a PyTorch Dataset from Hugging Face dataset
class EuroSATDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        super().__init__()
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        label = item["label"]

        if self.transform:
            image = self.transform(image)

        return image, label


# Define transformations
# Basic transformations for training and validation
basic_transform = transforms.Compose(
    [
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Augmented transformations for training
augmented_transform = transforms.Compose(
    [
        transforms.RandomRotation(20),
        transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Split the dataset into training, validation, and test sets
train_val_dataset = dataset["train"]  # type: ignore
test_dataset = dataset["test"]  # type: ignore

# Further split train_val into train and validation
train_hf_dataset, val_hf_dataset = random_split(
    train_val_dataset,  # type: ignore
    [0.8, 0.2],
    generator=torch.Generator().manual_seed(42),
)

# Create PyTorch datasets
train_dataset = EuroSATDataset(train_hf_dataset, transform=augmented_transform)
val_dataset = EuroSATDataset(val_hf_dataset, transform=basic_transform)
test_dataset = EuroSATDataset(test_dataset, transform=basic_transform)

# Create data loaders
batch_size = 64
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")

# Define a simple CNN model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            # First convolutional block
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Second convolutional block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Third convolutional block
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

simple_cnn = SimpleCNN(num_classes=len(class_names)).to(device)
print(simple_cnn)

# Define training and evaluation functions

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(
            train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"
        ):
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Statistics
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = 100.0 * correct / total
        train_losses.append(epoch_train_loss)
        train_accuracies.append(epoch_train_acc)

        # Validation phase
        model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in tqdm(
                val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"
            ):
                inputs, labels = inputs.to(device), labels.to(device)

                # Forward pass
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        epoch_val_loss = running_loss / len(val_loader.dataset)
        epoch_val_acc = 100.0 * correct / total
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        print(
            f"Epoch {epoch+1}/{num_epochs}: "
            f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, "
            f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%"
        )

    return train_losses, val_losses, train_accuracies, val_accuracies


def evaluate_model(model, test_loader, criterion, class_names):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating"):
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Statistics
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = running_loss / len(test_loader.dataset)
    test_acc = 100.0 * correct / total

    print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    # Plot confusion matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

    # Print classification report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    return test_loss, test_acc

# Train Simple CNN model    

In [ ]:
# Set up training parameters
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(simple_cnn.parameters(), lr=0.001)
num_epochs = 10

# Train the model
simple_cnn_history = train_model(
    simple_cnn, train_loader, val_loader, criterion, optimizer, num_epochs
)

# Unpack the history
train_losses, val_losses, train_accuracies, val_accuracies = simple_cnn_history

# Plot training and validation loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss Curves")

# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.title("Accuracy Curves")
plt.tight_layout()
plt.show()

# Evaluate Simple CNN model

In [ ]:
# Evaluate the model on the test set
simple_cnn_test_loss, simple_cnn_test_acc = evaluate_model(
    simple_cnn, test_loader, criterion, class_names
)

# Save the model
torch.save(simple_cnn.state_dict(), "simple_cnn_eurosat.pth")
print("Simple CNN model saved.")

# Use a ResNet18 Model

In [ ]:
# Load pre-trained ResNet18 model
resnet_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

print(resnet_model)

fc_in_features = resnet_model.fc.in_features
print(f"Number of input features in the fully connected layer: {fc_in_features}")

# Adapt model and fine-tune on the EuroSAT dataset (train)

In [ ]:
# Modify the final fully connected layer for our number of classes
resnet_model.fc = nn.Linear(fc_in_features, len(class_names))

# Move model to device
resnet_model = resnet_model.to(device)

# Freeze all layers except the final layer for fine-tuning
for param in resnet_model.parameters():
    param.requires_grad = False

# Unfreeze the final fully connected layer
for param in resnet_model.fc.parameters():
    param.requires_grad = True

# Set up training parameters
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_model.fc.parameters(), lr=0.001)
num_epochs = 10

# Train the model
resnet_history = train_model(
    resnet_model, train_loader, val_loader, criterion, optimizer, num_epochs
)

# Unpack the history
(
    resnet_train_losses,
    resnet_val_losses,
    resnet_train_accuracies,
    resnet_val_accuracies,
) = resnet_history

# Plot training and validation loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(resnet_train_losses, label="Train Loss")
plt.plot(resnet_val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("ResNet18 Loss Curves")

# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(resnet_train_accuracies, label="Train Accuracy")
plt.plot(resnet_val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.title("ResNet18 Accuracy Curves")
plt.tight_layout()
plt.show()

# Evaluate R18 model on the test set

In [ ]:
# Evaluate the model on the test set
resnet_test_loss, resnet_test_acc = evaluate_model(
    resnet_model, test_loader, criterion, class_names
)

# Save the model
torch.save(resnet_model.state_dict(), 'resnet18_eurosat.pth')
print("ResNet18 model saved.")

# Compare the 2 models

In [ ]:
# Compare the performance of both models
models = ['Simple CNN', 'ResNet18 (Fine-tuned)']
test_accuracies = [simple_cnn_test_acc, resnet_test_acc]

plt.figure(figsize=(10, 6))
plt.bar(models, test_accuracies, color=['blue', 'orange'])
plt.ylabel('Test Accuracy (%)')
plt.title('Model Comparison')
for i, v in enumerate(test_accuracies):
    plt.text(i, v + 1, f"{v:.2f}%", ha='center')
plt.ylim(0, 100)
plt.show()

# Print comparison summary
print("Model Comparison Summary:")
print(f"Simple CNN Test Accuracy: {simple_cnn_test_acc:.2f}%")
print(f"ResNet18 (Fine-tuned) Test Accuracy: {resnet_test_acc:.2f}%")
print(f"Improvement: {resnet_test_acc - simple_cnn_test_acc:.2f}%")

# Visualize predictions

In [ ]:
def visualize_predictions(model, test_loader, class_names, num_images=10):
    model.eval()
    images_so_far = 0
    plt.figure(figsize=(15, 10))

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size()[0]):
                if images_so_far >= num_images:
                    return

                images_so_far += 1
                ax = plt.subplot(2, 5, images_so_far)
                ax.axis('off')

                # Denormalize the image
                img = inputs.cpu()[j].numpy().transpose((1, 2, 0))
                mean = np.array([0.485, 0.456, 0.406])
                std = np.array([0.229, 0.224, 0.225])
                img = std * img + mean
                img = np.clip(img, 0, 1)

                ax.imshow(img)

                # Color-code the title based on prediction correctness
                title_color = 'green' if preds[j] == labels[j] else 'red'
                ax.set_title(f'True: {class_names[labels[j]]}\nPred: {class_names[preds[j]]}',
                             color=title_color)

# Visualize predictions from both models
print("Simple CNN Predictions:")
visualize_predictions(simple_cnn, test_loader, class_names)
plt.tight_layout()
plt.show()

print("\nResNet18 Predictions:")
visualize_predictions(resnet_model, test_loader, class_names)
plt.tight_layout()
plt.show()